In [1]:
import os
import glob
import rasterio
import geopandas as gpd
'''
# 1. Load the shapefile directly (GeoPandas automatically reads the UTM geometry and CRS)
# points_gdf = gpd.read_file("C:/Users/arj26323/Documents/Data/Pennings_validation_points/gce_sites_zone_pixels_centroids/gce_sites_zone_pixels_centroids.shp")
# points_gdf = gpd.read_file("C:/Users/arj26323/Documents/Data/Pennings_validation_points/gce_sites_zone_pixels_centroids2/gce_sites_zone_pixels_centroids2.shp")

# 2. Get the native CRS of your biomass models
raster_files = sorted(glob.glob('F:/Biomass tiffs/Biomass modeling/Georgia/Georgia model only/LOGO_final/*.tif'))
with rasterio.open(raster_files[0]) as src:
    raster_crs = src.crs

# 3. Project the points to match the raster if they don't already
if points_gdf.crs != raster_crs:
    points_gdf = points_gdf.to_crs(raster_crs)

# Extract coordinates as a list of (x, y) tuples
coord_tuples = [(pt.x, pt.y) for pt in points_gdf.geometry]

# 4. Loop through the years and sample the data
for raster_path in raster_files:
    year = os.path.basename(raster_path).split("_")[-1].replace(".tif", "")
    print(f"Extracting data for year: {year}...")
    
    with rasterio.open(raster_path) as src:
        # Sample the raster and append directly to the GeoDataFrame
        points_gdf[f"biomass_{year}"] = [val[0] for val in src.sample(coord_tuples)]

# 5. Save out the clean results
# You can save it back out as a flat CSV for them, or even a new shapefile/GeoJSON
points_gdf.drop(columns='geometry').to_csv("biomass_extractions_final.csv", index=False)
print("Done!")'''''

'\n# 1. Load the shapefile directly (GeoPandas automatically reads the UTM geometry and CRS)\n# points_gdf = gpd.read_file("C:/Users/arj26323/Documents/Data/Pennings_validation_points/gce_sites_zone_pixels_centroids/gce_sites_zone_pixels_centroids.shp")\n# points_gdf = gpd.read_file("C:/Users/arj26323/Documents/Data/Pennings_validation_points/gce_sites_zone_pixels_centroids2/gce_sites_zone_pixels_centroids2.shp")\n\n# 2. Get the native CRS of your biomass models\nraster_files = sorted(glob.glob(\'F:/Biomass tiffs/Biomass modeling/Georgia/Georgia model only/LOGO_final/*.tif\'))\nwith rasterio.open(raster_files[0]) as src:\n    raster_crs = src.crs\n\n# 3. Project the points to match the raster if they don\'t already\nif points_gdf.crs != raster_crs:\n    points_gdf = points_gdf.to_crs(raster_crs)\n\n# Extract coordinates as a list of (x, y) tuples\ncoord_tuples = [(pt.x, pt.y) for pt in points_gdf.geometry]\n\n# 4. Loop through the years and sample the data\nfor raster_path in raster_fi

In [2]:
points_gdf

NameError: name 'points_gdf' is not defined

In [3]:
##Exporting extraction

out_dir = os.path.expanduser('~/Downloads')
out_csv = os.path.join(out_dir, 'rest_trends.csv')

#points_gdf.to_csv(out_csv)

In [11]:
##Window extraction around restoration points
import rasterio
from rasterio.windows import Window
import numpy as np
import pandas as pd
import geopandas as gpd
import glob
import os

# 1. Setup
'''points_gdf = gpd.read_file("C:/Users/arj26323/Documents/Data/Pennings_validation_points/Restoration sites/Rest_sites.shp")
'''
raster_files = sorted(glob.glob('F:/Biomass tiffs/Biomass modeling/Georgia/Georgia model only/LOGO_final/*.tif'))

# Load CSV
csv_path = "C:/Users/arj26323/Documents/Data/GA_conservation/fixed_actives_installations.csv" #UPDATED LIVINGSHORELINE LOCATIONS
df = pd.read_csv(csv_path)

# Create GeoDataFrame
points_gdf = gpd.GeoDataFrame(
    df, 
    geometry=gpd.points_from_xy(df.Longitude, df.Latitude),
    crs="EPSG:4326"
)

'''with rasterio.open(raster_files[0]) as src:
    raster_crs = src.crs
    if points_gdf.crs != raster_crs:
        points_gdf = points_gdf.to_crs(raster_crs)'''

# Define window size (e.g., 2x2)
win_size = 5
pad = win_size // 5

# 2. Loop through files
for raster_path in raster_files:
    year = os.path.basename(raster_path).split("_")[-1].replace(".tif", "")
    print(f"Extracting window mean for year: {year}...")
    
    with rasterio.open(raster_path) as src:
        results = []
        for idx, row in points_gdf.iterrows():
            # Get pixel coordinates
            row_idx, col_idx = src.index(row.geometry.x, row.geometry.y)
            
            # Define the window
            window = Window(col_idx - pad, row_idx - pad, win_size, win_size)
            
            # Read the window data
            data = src.read(1, window=window)
            
            # Mask out nodata/NaN values
            # Replace 'src.nodata' with np.nan if your tiff uses actual NaNs
            if src.nodata is not None:
                data = np.where(data == src.nodata, np.nan, data)
                
            # Calculate mean ignoring NaNs
            if np.all(np.isnan(data)):
                results.append(np.nan)
            else:
                results.append(np.nanmean(data))
        
        points_gdf[f"biomass_{year}"] = results

# 3. Save
points_gdf.drop(columns='geometry').to_csv("biomass_extractions_5x5.csv", index=False)
print("Done!")

Extracting window mean for year: 1984...
Extracting window mean for year: 1985...
Extracting window mean for year: 1986...
Extracting window mean for year: 1987...
Extracting window mean for year: 1988...
Extracting window mean for year: 1989...
Extracting window mean for year: 1990...
Extracting window mean for year: 1991...
Extracting window mean for year: 1992...
Extracting window mean for year: 1993...
Extracting window mean for year: 1994...
Extracting window mean for year: 1995...
Extracting window mean for year: 1996...
Extracting window mean for year: 1997...
Extracting window mean for year: 1998...
Extracting window mean for year: 1999...
Extracting window mean for year: 2000...
Extracting window mean for year: 2001...
Extracting window mean for year: 2002...
Extracting window mean for year: 2003...
Extracting window mean for year: 2004...
Extracting window mean for year: 2005...
Extracting window mean for year: 2006...
Extracting window mean for year: 2007...
Extracting windo

In [12]:
points_gdf

,Latitude,Longitude,Site name,year,type,geometry,biomass_1984,biomass_1985,biomass_1986,biomass_1987,...,biomass_2014,biomass_2015,biomass_2016,biomass_2017,biomass_2018,biomass_2019,biomass_2020,biomass_2021,biomass_2022,biomass_2023
0,31.122967,-81.477183,Plantation Creek Site,2010,Oyster Reef,POINT (-81.47718 31.12297),359.861572,517.220154,367.110535,404.826416,...,396.410889,575.266785,413.262115,516.503967,399.500885,367.425171,614.039307,633.087830,423.380432,406.540436
1,31.838650,-81.153333,Florida Passage Site,2012,Oyster Reef,POINT (-81.15333 31.83865),467.979218,520.887451,652.131104,520.747253,...,390.632751,640.385864,463.076721,470.295624,468.255371,412.596252,540.196594,469.956512,448.260498,487.868439
2,32.048017,-81.019233,Oatland Site,2014,Oyster Reef,POINT (-81.01923 32.04802),466.399323,424.682587,603.626221,472.495361,...,426.259766,631.455261,427.532898,446.567413,474.296844,398.815399,626.939758,609.911072,420.730377,435.374481
3,31.988800,-81.024883,Skidaway River Site,2010,Oyster Reef,POINT (-81.02488 31.9888),NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,31.326150,-81.322283,Estuaries Area of the Altamaha River,2009,Oyster Reef,POINT (-81.32228 31.32615),429.707520,415.862732,341.976624,362.343506,...,1787.974243,1959.631104,1678.214478,1945.666992,1816.617188,1507.119751,1955.317139,1914.513306,1738.216797,1590.750732
5,31.041650,-81.423017,Jekyll Island Boat Ramp Site,2009,Oyster Reef,POINT (-81.42302 31.04165),525.224426,729.417542,580.894226,615.125427,...,699.967163,800.085632,681.284851,716.593872,733.602478,723.067261,762.305176,831.625488,639.630005,795.828857
6,32.002983,-80.917817,Oyster Creek Site A,2013,Oyster Reef,POINT (-80.91782 32.00298),283.327515,397.633209,396.362396,276.705017,...,263.679810,373.867554,239.567932,261.988892,247.395554,251.050980,257.231934,257.997437,273.876099,253.431091
7,32.004417,-80.914883,Oyster Creek Site B,2013,Oyster Reef,POINT (-80.91488 32.00442),263.357117,400.524597,376.023254,259.259338,...,243.975906,375.204315,196.735092,227.559311,204.722626,238.361404,241.066498,207.819061,199.367325,210.451843
8,32.007950,-80.911683,Oyster Creek Site C,2013,Oyster Reef,POINT (-80.91168 32.00795),303.001709,394.366333,393.221710,328.405792,...,242.188736,352.372162,208.212662,224.389801,225.854935,225.366119,238.843323,213.525497,218.763870,219.447769
9,31.080800,-81.500217,Jointer Creek Oyster Reef A,2012,Oyster Reef,POINT (-81.50022 31.0808),290.034821,413.622101,315.562347,350.982605,...,312.176910,473.172668,318.294128,427.311829,310.625824,279.873993,450.481995,656.532227,312.164337,334.769592


In [13]:
##Exporting extraction

out_dir = os.path.expanduser('~/Downloads')
out_csv = os.path.join(out_dir, 'fixed_restoration_window_trends_5.csv')

# points_gdf.to_csv(out_csv)

In [13]:
# 1. Setup

import geopandas as gpd
import rasterio
from rasterio.mask import mask
import numpy as np
import os
import glob
import pandas as pd

# polygons = gpd.read_file('F:/Hotspots/Project shapefiles/hotspots_theilsen_polyg_500k_m2.shp')
# polygons = gpd.read_file('F:/Hotspots/Project shapefiles/hotspots_theilsen_savan_100_m2.shp')
# polygons = gpd.read_file('F:/Hotspots/Project shapefiles/hotspot_jekyllish.shp')
# polygons = gpd.read_file('F:/Hotspots/Project shapefiles/hotspots_theilsen_sap_100.shp')
polygons = gpd.read_file('F:/Hotspots/Project shapefiles/all_100.shp')

raster_files = sorted(glob.glob('F:/Biomass tiffs/Biomass modeling/Georgia/Georgia model only/LOGO_final/*.tif'))

# Initialize an empty list to store results for each polygon
results = []

# 2. Iterate through each raster
for raster_path in raster_files:
    year = os.path.basename(raster_path).split("_")[-1].replace(".tif", "")
    print(f"Processing {year}...")
    
    with rasterio.open(raster_path) as src:
        # Ensure CRS matches
        if polygons.crs != src.crs:
            polygons = polygons.to_crs(src.crs)
            
        # Loop through each polygon
        for idx, row in polygons.iterrows():
            # Use rasterio.mask to clip the raster to the polygon geometry
            # crop=True keeps the resulting array small (the bounding box of the polygon)
            out_image, out_transform = mask(src, [row.geometry], crop=True, nodata=np.nan)
            
            # The mask returns an array [band, height, width]
            data = out_image[0]
            
            # Remove NaNs (NoData pixels)
            valid_pixels = data[~np.isnan(data)]
            
            # Calculate stats
            if valid_pixels.size > 0:
                mean_val = np.nanmean(valid_pixels)
                std_val = np.nanstd(valid_pixels)
            else:
                mean_val = np.nan
                std_val = np.nan
                
            results.append({
                'polygon_id': idx,
                'year': year,
                'mean': mean_val,
                'std': std_val
            })

# 3. Convert results to a DataFrame
stats_df = pd.DataFrame(results)

# Optional: Pivot to wide format if you want one column per year
pivot_df = stats_df.pivot(index='polygon_id', columns='year', values='mean')
print("Extraction complete!")

Processing 1984...
Processing 1985...
Processing 1986...
Processing 1987...
Processing 1988...
Processing 1989...
Processing 1990...
Processing 1991...
Processing 1992...
Processing 1993...
Processing 1994...
Processing 1995...
Processing 1996...
Processing 1997...
Processing 1998...
Processing 1999...
Processing 2000...
Processing 2001...
Processing 2002...
Processing 2003...
Processing 2004...
Processing 2005...
Processing 2006...
Processing 2007...
Processing 2008...
Processing 2009...
Processing 2010...
Processing 2011...
Processing 2012...
Processing 2013...
Processing 2014...
Processing 2015...
Processing 2016...
Processing 2017...
Processing 2018...
Processing 2019...
Processing 2020...
Processing 2021...
Processing 2022...
Processing 2023...
Extraction complete!


In [14]:
stats_df

,polygon_id,year,mean,std
0,0,1984,465.578949,34.439026
1,1,1984,465.542297,33.199669
2,2,1984,476.600403,42.166412
3,3,1984,456.660065,25.501823
4,4,1984,418.759491,69.571579
...,...,...,...,...
3715,88,2023,345.837524,270.438843
3716,89,2023,249.398560,40.118248
3717,90,2023,399.588562,355.779602
3718,91,2023,465.820587,147.515091


In [15]:
##Exporting extraction

out_dir = os.path.expanduser('~/Downloads')
out_csv = os.path.join(out_dir, 'all_10hectare_hotspots_trends.csv')

#stats_df.to_csv(out_csv)

In [ ]:
#Fin